# 20 - vLLM latest baseline (its own notebook: it installs its own torch)

In [ ]:
!pip install -q vllm
!python -c "import vllm, torch; print(vllm.__version__, torch.__version__)"
!hf download Qwen/Qwen2.5-1.5B-Instruct --local-dir /kaggle/working/models/Qwen2.5-1.5B-Instruct

In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/vllm.log", "w")
p = subprocess.Popen(["vllm", "serve", "/kaggle/working/models/Qwen2.5-1.5B-Instruct", "--served-model-name", "qwen",
                      "--dtype", "half", "--max-model-len", "4096", "--gpu-memory-utilization", "0.85",
                      "--port", "8001"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if requests.get("http://localhost:8001/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(5)
print(requests.post("http://localhost:8001/v1/completions",
                    json={"model": "qwen", "prompt": "Hello", "max_tokens": 16}).json())

In [ ]:
!grep -i -E "backend|attention" /kaggle/working/vllm.log | head -20

Record the vLLM version, torch version, the attention backend and whether it works in `docs/baselines.md` (paste the output of the two cells above). Then run the SAME workload as the TuringServe sweep against port 8001: ShareGPT, 500 prompts, concurrency 1,4,16,32,64. Do not `pip install` torch-related packages here; vLLM brought its own.

In [ ]:
!git clone https://github.com/Muskanpandey0304/turingserve.git /kaggle/working/turingserve
%cd /kaggle/working/turingserve
!git checkout m3-rc
!pip install -q httpx
!hf download anon8231489123/ShareGPT_Vicuna_unfiltered ShareGPT_V3_unfiltered_cleaned_split.json --repo-type dataset --local-dir /kaggle/working/datasets/sharegpt


In [ ]:
import os
os.environ["TS_SHAREGPT"] = "/kaggle/working/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
os.environ["TS_KAGGLE"] = "1"
r = subprocess.run(["python", "-m", "benchmarks.serve_bench", "--base-url", "http://localhost:8001", "--model", "qwen",
                    "--tokenizer", "/kaggle/working/models/Qwen2.5-1.5B-Instruct", "--dataset", "sharegpt", "--num-prompts", "500",
                    "--mode", "concurrency", "--concurrency", "1,4,16,32,64", "--ignore-eos", "--label", "t4-1.5b",
                    "--out", "/kaggle/working/results"], capture_output=True, text=True)
print(r.stdout[-2500:]); print(r.stderr[-1500:] if r.returncode else "")


Download the `*_vllm_sharegpt_t4-1.5b.json` file from `/kaggle/working/results/` and `vllm.log`. If the sweep errors, paste the output. Prefill (2k/4k/8k) needs a restart with `--max-model-len 8192`: terminate the server below, change that value in the launch cell and rerun it.

In [ ]:
p.terminate(); p.wait()
